# Day 7: Incremental Processing & CDC Upsert Engine

This notebook demonstrates idempotent UPSERT/DELETE logic on a simulated stateful Iceberg table using MERGE INTO.

In [ ]:
import os
from pyspark.sql import SparkSession

os.environ['SPARK_VERSION'] = '3.5'

spark = SparkSession.builder \
    .appName("Day07_Incremental_Merge") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


In [ ]:
# Create Target Table with Initial Data
spark.sql("CREATE DATABASE IF NOT EXISTS local.silver_db")
spark.sql("""
CREATE OR REPLACE TABLE local.silver_db.patient_state (
    patient_id STRING,
    status STRING,
    updated_at TIMESTAMP,
    op STRING
) USING iceberg
""")

spark.sql("""
INSERT INTO local.silver_db.patient_state VALUES 
    ('P1', 'active', timestamp '2023-01-01 10:00:00', 'c'),
    ('P2', 'active', timestamp '2023-01-01 10:00:00', 'c')
""")

print("Initial State:")
spark.sql("SELECT * FROM local.silver_db.patient_state").show()


In [ ]:
# Incoming CDC Data (Simulated)
# - P1 gets updated to inactive
# - P2 gets deleted
# - P3 is a new creation
# - P1 out-of-order old update (should be ignored)
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW cdc_updates AS SELECT * FROM VALUES
    ('P1', 'inactive', timestamp '2023-01-02 10:00:00', 'u'),
    ('P2', 'active', timestamp '2023-01-02 10:00:00', 'd'),
    ('P3', 'active', timestamp '2023-01-02 10:00:00', 'c'),
    ('P1', 'pending', timestamp '2022-12-31 10:00:00', 'u') 
    AS (patient_id, status, updated_at, op)
""")

print("Incoming CDC Updates:")
spark.sql("SELECT * FROM cdc_updates").show()


In [ ]:
# Execute MERGE INTO
merge_query = """
MERGE INTO local.silver_db.patient_state t
USING cdc_updates s
ON t.patient_id = s.patient_id
WHEN MATCHED AND s.op = 'd' AND s.updated_at >= t.updated_at THEN
    DELETE
WHEN MATCHED AND s.op IN ('u', 'c') AND s.updated_at > t.updated_at THEN
    UPDATE SET *
WHEN NOT MATCHED AND s.op IN ('c', 'u') THEN
    INSERT *
"""

spark.sql(merge_query)

print("Final State After Incremental Merge:")
spark.sql("SELECT * FROM local.silver_db.patient_state ORDER BY patient_id").show()
